> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# Parameter search for the intraday momentum strategy on WIN and WDO

A random search of the eight parameters the owner approved (`docs/brief.md` section 10), run on the **WIN mini index** and the **WDO mini dollar**, scored three ways: WIN alone, WDO alone, and **both with shared parameters** (the worse of the two Sharpe ratios). Development period only (2021-12-01 to 2024-12-30, 770 days); the final test period is **not** loaded.

**Read this before any number**
- **The best of many trials is partly luck.** With about 750 days a Sharpe ratio has a standard error near 0.58, and the best of N unrelated trials is expected to reach roughly `0.58 x sqrt(2 ln N)` with no edge at all: about 2.0 for 500 trials, 2.3 for 3,000 and 2.7 for 50,000. Only the held-out numbers count: the leave-one-year-out check, the transfer from one instrument to the other, and the comparison with the **luck baseline** at the end.
- **Costs are in the loop.** Each side pays the B3 fee **plus one tick of slippage**: WIN R$ 0.25 + R$ 1.00, WDO R$ 1.20 + R$ 5.00 per contract and side. Relative to what a contract controls, WDO's costs are about twice WIN's.
- **The sizing is fixed** at the paper's volatility target of 2% and leverage cap of 4; the margin is R$ 1,000 a contract; each instrument runs on its own R$ 100,000 account.
- **The trials are shared.** The same combinations are evaluated once on each instrument; every mode and every fold is scored from the stored daily returns. A day without a sigma (the first days of a long lookback) counts as a flat day. The per-year scores reuse the full-period equity path, which only affects the rounding to whole contracts.
- **Nothing here is chosen for the final test.** This notebook shows how the search behaves.

In [ ]:
import multiprocessing as mp
import os
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from tqdm import tqdm

from b3core.intraday_search import (
    INSTRUMENTS,
    SPACE,
    TrialSet,
    decode,
    encode,
    fold_of,
    leave_one_year_out,
    make_config,
    neighbours,
    portfolio_sharpe,
    run_trial,
    sample_indices,
    scores,
    scramble_bars,
    space_size,
    stack_trials,
)
from b3core.winfut import FINAL_START, load_bars

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 230
pd.options.display.max_columns = 40
pd.options.display.max_colwidth = 110

## 1. Settings

In [ ]:
N_TRIALS = int(os.environ.get("N_TRIALS", 3_000))  # the budget (3,000 by default; set the variable to change it)
SEED = 20261116  # the sample of combinations, and (plus one) the scrambled data
SLIPPAGE_TICKS = 1.0  # per side, on top of the B3 fee
MIN_TRADES = 200  # round trips in a training fold (two years); a trial with fewer is not ranked
N_NULL = int(os.environ.get("N_NULL", 500))  # trials of the luck baseline
WORKERS = max(1, min(12, (os.cpu_count() or 2) - 1))
MIN_TRADES_FULL = round(MIN_TRADES * 1.5)  # the same minimum scaled to the three years
MODES = ["WIN", "WDO", "joint"]
print(f"{N_TRIALS:,} trials, seed {SEED}, {WORKERS} worker processes, slippage {SLIPPAGE_TICKS:g} tick a side")

## 2. The parameter space and the costs

In [ ]:
def clock(minutes: int) -> str:
    return "session end" if minutes >= 1440 else f"{minutes // 60:02d}:{minutes % 60:02d}"


rows = []
for name, values in SPACE.items():
    shown = [clock(v) if name in ("first_decision", "exit_minute") else v for v in values]
    rows.append({"parameter": name, "values": ", ".join(str(v) for v in shown), "count": len(values)})
space_table = pd.DataFrame(rows).set_index("parameter")
display(space_table)
print(f"{space_size():,} combinations in all; {N_TRIALS:,} sampled ({N_TRIALS / space_size():.2%}).")
print("Fixed at the paper's values: target volatility 2%, leverage cap 4, a 14-day volatility window. Not searched: capital, point values, fees, margin.")

bars = {name: load_bars("../samples", symbol=inst.symbol) for name, inst in INSTRUMENTS.items()}
for frame in bars.values():
    assert frame.index.max() < pd.Timestamp(FINAL_START.date()), "a bar from the final test period was loaded"
days = pd.DatetimeIndex(sorted(set.intersection(*[set(f.index.normalize().unique()) for f in bars.values()])))
opens = {n: f["open"].groupby(f.index.normalize()).first().mean() for n, f in bars.items()}
cost_rows = {}
for name, inst in INSTRUMENTS.items():
    exposure = opens[name] * inst.point_value
    side = inst.fee_per_side + SLIPPAGE_TICKS * inst.tick_reais
    cost_rows[name] = {"point value, R$": inst.point_value, "fee a side, R$": inst.fee_per_side, "one tick, R$": inst.tick_reais,
                       "cost a side, R$": side, "contract exposure, R$": exposure, "round trip, basis points": 2 * side / exposure * 1e4}
display(pd.DataFrame(cost_rows).T)
print(f"{len(days)} common days, {days[0].date()} to {days[-1].date()}; bars: " + ", ".join(f"{n} {len(f):,}" for n, f in bars.items()))

## 3. The search

Each trial runs one combination on both instruments. The pool uses every core but one. The bar shows the rate and the seconds of the last trial; the table after it gives the timing of the whole run: the wall time, the time per trial, and how much the pool saved.

In [ ]:
_STATE: dict = {}


def work(index: int):
    started = time.perf_counter()
    returns, sides = run_trial(_STATE["bars"], decode(index), days, slippage_ticks=_STATE["slippage"])
    return index, returns, sides, time.perf_counter() - started


def search(indices: list[int], data: dict, label: str, slippage: float = SLIPPAGE_TICKS):
    _STATE.update(bars=data, slippage=slippage)
    position = {index: i for i, index in enumerate(indices)}
    results: list = [None] * len(indices)
    seconds = np.zeros(len(indices))
    started = time.perf_counter()
    with ProcessPoolExecutor(max_workers=WORKERS, mp_context=mp.get_context("fork")) as pool:
        futures = [pool.submit(work, index) for index in indices]
        done_seconds: list[float] = []
        with tqdm(total=len(indices), desc=label, unit="trial", mininterval=2.0) as bar:
            for future in as_completed(futures):
                index, returns, sides, took = future.result()
                results[position[index]] = (index, returns, sides)
                seconds[position[index]] = took
                done_seconds.append(took)
                bar.update(1)
                bar.set_postfix(last=f"{took:.2f}s", mean=f"{np.mean(done_seconds):.2f}s", refresh=False)
    wall = time.perf_counter() - started
    timing = pd.Series(
        {
            "trials": len(indices),
            "wall time, s": wall,
            "trials per second": len(indices) / wall,
            "seconds per trial, mean (one core, both instruments)": seconds.mean(),
            "seconds per trial, median": np.median(seconds),
            "seconds per trial, 95th percentile": np.percentile(seconds, 95),
            "seconds per trial, slowest": seconds.max(),
            "time of all trials added up, s": seconds.sum(),
            "speed-up from the pool": seconds.sum() / wall,
            "workers": WORKERS,
        },
        name=label,
    )
    return stack_trials(results, days), seconds, timing


indices = sample_indices(N_TRIALS, SEED)
trials, trial_seconds, timing = search(indices, bars, "search")
display(timing.to_frame())

from pathlib import Path

SAVE = Path("../samples/optimization")  # gitignored: derived from vendor data
SAVE.mkdir(parents=True, exist_ok=True)
np.savez_compressed(SAVE / f"trials_{N_TRIALS}_seed{SEED}.npz", indices=trials.indices, days=days.to_numpy().astype("datetime64[ns]"),
                    seconds=trial_seconds, **{f"ret_{k}": v for k, v in trials.ret.items()}, **{f"sides_{k}": v for k, v in trials.sides.items()})
print(f"trial results saved to {SAVE}/trials_{N_TRIALS}_seed{SEED}.npz")

**Seconds per trial by parameter.** Which choices make a trial slow (the every-bar stop and the VWAP-only stop are bar-by-bar state machines; the half-hour grid is vectorised).

In [ ]:
params = pd.DataFrame([decode(i) for i in trials.indices])
params["exit_minute"] = params["exit_minute"].map(clock)
params["first_decision"] = params["first_decision"].map(clock)
slow = pd.DataFrame({"stop": params["stop"], "stop_check": params["stop_check"], "seconds": trial_seconds})
display(slow.groupby(["stop", "stop_check"])["seconds"].agg(["count", "mean", "max"]).round(2))

## 4. Reference runs

The paper's rule, and two variants already looked at, scored by the same code and with the same costs (fee and one tick). They are extra runs, not part of the trials.

In [ ]:
PAPER = {"lookback": 14, "band_mult": 1.0, "trade_freq": 30, "first_decision": 540, "exit_minute": 1440,
         "stop": "vwap_band", "stop_check": "grid", "sides": "both"}
REFERENCE = {
    "paper (no exit time, half-hour stop)": PAPER,
    "exit at 17:30": PAPER | {"exit_minute": 1050},
    "every-bar stop": PAPER | {"stop_check": "every_bar"},
}
ref_indices = [encode(p) for p in REFERENCE.values()]
refs, _, _ = search(ref_indices, bars, "reference runs")
EVERYTHING = np.ones(len(days), dtype=bool)
FULL = {m: scores(trials, EVERYTHING, m, MIN_TRADES_FULL) for m in MODES}
ref_table = pd.DataFrame({m: scores(refs, EVERYTHING, m, 0) for m in MODES}, index=list(REFERENCE))
ref_table["portfolio 50/50"] = portfolio_sharpe(refs, EVERYTHING)
display(ref_table.rename(columns={"joint": "joint (worse of the two)"}))

## 5. How the trials scored over the whole development period

Sharpe ratio of the daily net returns, costs included. `joint` is the worse of the two instruments' Sharpe ratios; a trial needs at least 300 round trips on each instrument it is scored on. The vertical line is the paper's rule.

In [ ]:
rows = {}
for m in MODES:
    s = FULL[m]
    ok = s[np.isfinite(s)]
    rows[m] = {"trials ranked": len(ok), "share > 0": (ok > 0).mean() * 100, "median": np.median(ok), "95th percentile": np.percentile(ok, 95),
               "99th percentile": np.percentile(ok, 99), "best": ok.max(), "paper's rule": ref_table[m if m != "joint" else "joint"].iloc[0]}
display(pd.DataFrame(rows).T)
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{m}" for m in MODES])
for col, m in enumerate(MODES, start=1):
    s = FULL[m]
    fig.add_trace(go.Histogram(x=s[np.isfinite(s)], nbinsx=50, showlegend=False), row=1, col=col)
    fig.add_vline(x=float(ref_table[m].iloc[0]), line_dash="dash", line_color="black", row=1, col=col)
fig.update_layout(title="Sharpe ratio of the trials (dashed: the paper's rule)", height=340, margin=dict(t=70, b=20))
fig.show()

## 6. The best trials of each mode

The ten best by score over the whole period. The scores here are in-sample: they are inflated by the search, which is the point of the next sections.

In [ ]:
def describe(index: int) -> str:
    p = decode(index)
    return (f"{p['lookback']}d x{p['band_mult']:g} every {p['trade_freq']}m from {clock(p['first_decision'])} "
            f"exit {clock(p['exit_minute'])} {p['stop']}/{p['stop_check']} {p['sides']}")


portfolio_all = portfolio_sharpe(trials, EVERYTHING)
win_all, wdo_all = scores(trials, EVERYTHING, "WIN", 0), scores(trials, EVERYTHING, "WDO", 0)
round_trips = {n: trials.sides[n].sum(axis=1) / 2 for n in ("WIN", "WDO")}


def top_table(mode: str, count: int = 10) -> pd.DataFrame:
    s = FULL[mode]
    order = np.argsort(np.where(np.isfinite(s), s, -np.inf))[::-1][:count]
    return pd.DataFrame(
        {"score": s[order], "WIN Sharpe": win_all[order], "WDO Sharpe": wdo_all[order], "50/50 Sharpe": portfolio_all[order],
         "WIN trades": round_trips["WIN"][order], "WDO trades": round_trips["WDO"][order],
         "combination": [describe(int(trials.indices[i])) for i in order]},
        index=pd.RangeIndex(1, len(order) + 1, name="rank"),
    )


for m in MODES:
    print(f"Best by {m}")
    display(top_table(m))

## 7. Leave one year out

The honest test inside the development period. For each calendar year (December 2021 joins 2022): pick the best trial on the **other two years**, with at least 200 round trips there, then score it on the year held out. `median test` is what a random pick would get that year; `percentile` is where the winner's held-out score falls among all trials' held-out scores; `paper's rule` is the paper's configuration on the same year. A winner that is great in training and ordinary or worse held out is the signature of overfitting.

In [ ]:
folds = fold_of(days)
loyo = {}
for m in MODES:
    table = leave_one_year_out(trials, m, MIN_TRADES)
    paper_test = [float(scores(refs, folds == y, m, 0)[0]) for y in table["held out"]]
    table["paper's rule"] = paper_test
    table["winner"] = [describe(int(trials.indices[int(w)])) if w == w else "none passes the trade minimum" for w in table["winner"]]
    loyo[m] = table
    print(f"Mode {m}")
    display(table.set_index("held out"))

summary = pd.DataFrame(
    {m: {"mean training score of the winners": t["train"].mean(), "mean held-out score of the winners": t["test"].mean(),
         "mean held-out median trial": t["median test"].mean(), "mean held-out paper's rule": t["paper's rule"].mean(),
         "gap, training minus held out": t["train"].mean() - t["test"].mean()} for m, t in loyo.items()}
).T
print("Summary across the three held-out years")
display(summary)

## 8. Does a winner on one instrument work on the other?

The best trial of WIN, scored on WDO, and the reverse, with each score placed among the other instrument's own scores. The days are the same, so this is not independent in time, but it is a different market with different costs. The joint winner is shown for comparison.

In [ ]:
def transfer(source: str, target: str) -> dict:
    s = FULL[source]
    best = int(np.nanargmax(np.where(np.isfinite(s), s, -np.inf)))
    target_scores = scores(trials, EVERYTHING, target, 0) if target != "joint" else FULL["joint"]
    value = target_scores[best]
    finite = target_scores[np.isfinite(target_scores)]
    return {"best trial of": source, "scored on": target, "its score there": value,
            "percentile among that mode's trials": (finite <= value).mean() * 100, "the target's own best": finite.max(),
            "combination": describe(int(trials.indices[best]))}


display(pd.DataFrame([transfer("WIN", "WDO"), transfer("WDO", "WIN"), transfer("joint", "WIN"), transfer("joint", "WDO")]))

## 9. Which parameters matter

For each value of each parameter, the mean joint score of the trials that use it, and the share of the best 5% of trials (by joint score) that use it against the share in all trials. A parameter whose values all look alike has no effect the search can see; a value that is over-represented among the best is where the search finds something.

In [ ]:
joint = FULL["joint"]
ranked = np.isfinite(joint)
top = ranked & (joint >= np.nanpercentile(joint, 95))
raw = pd.DataFrame([decode(i) for i in trials.indices])
def mean_or_nan(values):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(values)


rows = []
for name in SPACE:
    for value in SPACE[name]:
        has = (raw[name] == value).to_numpy()
        rows.append({"parameter": name, "value": clock(value) if name in ("first_decision", "exit_minute") else value,
                     "trials": int(has.sum()), "mean joint score": mean_or_nan(np.where(has & ranked, joint, np.nan)),
                     "share of all trials %": has.mean() * 100, "share of the best 5% %": (has & top).sum() / max(top.sum(), 1) * 100})
effects = pd.DataFrame(rows).set_index(["parameter", "value"])
display(effects)
fig = make_subplots(rows=2, cols=4, subplot_titles=list(SPACE), vertical_spacing=0.18)
for k, name in enumerate(SPACE):
    sub = effects.loc[name]
    fig.add_trace(go.Bar(x=[str(v) for v in sub.index], y=sub["mean joint score"], showlegend=False), row=k // 4 + 1, col=k % 4 + 1)
fig.update_layout(title="Mean joint score by parameter value", height=520, margin=dict(t=80, b=20))
fig.show()

## 10. The neighbourhood of the best joint trial

A good combination surrounded by bad neighbours is a warning; a plateau is a good sign. The best joint trial and every combination one step away on one parameter (a step is one value of that parameter's list), run on both instruments with the same costs.

In [ ]:
best_joint = int(np.nanargmax(np.where(np.isfinite(joint), joint, -np.inf)))
best_index = int(trials.indices[best_joint])
around = neighbours(best_index)
near, _, near_timing = search([best_index] + around, bars, "neighbourhood")
near_scores = {m: scores(near, EVERYTHING, m, 0) for m in MODES}
table = pd.DataFrame({"WIN Sharpe": near_scores["WIN"], "WDO Sharpe": near_scores["WDO"], "joint score": near_scores["joint"],
                      "changed": ["(the best trial)"] + [", ".join(f"{n} {a} -> {b}" for n, a, b in
                                                                 [(n, decode(best_index)[n], decode(i)[n]) for n in SPACE if decode(i)[n] != decode(best_index)[n]])
                                                         for i in around]})
display(table)
neighbours_joint = near_scores["joint"][1:]
print(f"best joint score {near_scores['joint'][0]:.2f}; neighbours: median {np.nanmedian(neighbours_joint):.2f}, worst {np.nanmin(neighbours_joint):.2f}, "
      f"{(neighbours_joint >= 0.8 * near_scores['joint'][0]).sum()} of {len(neighbours_joint)} keep at least 80% of it")

## 11. The luck baseline

The identical search on the same data with the order of each day's moves **shuffled** (`scramble_bars`): every day keeps its open, its last close, its volume and the set of its moves, but any dependence between one move and the next, and the time-of-day shape of the volatility, is gone, so no momentum edge can exist. The first `N_NULL` of the same combinations are run on both instruments. If the real search's best score is no higher than what pure noise reaches, the best trial is not telling us anything.

In [ ]:
scrambled = {name: scramble_bars(frame, SEED + 1) for name, frame in bars.items()}
null_trials, _, null_timing = search(indices[:N_NULL], scrambled, "luck baseline")
display(null_timing.to_frame())
NULL_FULL = {m: scores(null_trials, EVERYTHING, m, MIN_TRADES_FULL) for m in MODES}
rows = {}
for m in MODES:
    real_all = FULL[m]
    real = real_all[np.isfinite(real_all)]
    real_500 = FULL[m][: N_NULL][np.isfinite(FULL[m][: N_NULL])]
    null = NULL_FULL[m][np.isfinite(NULL_FULL[m])]
    rows[m] = {f"real: best of the same {N_NULL:,}": real_500.max(), f"scrambled: best of {N_NULL:,}": null.max(), "real: 95th percentile": np.percentile(real_500, 95),
               "scrambled: 95th percentile": np.percentile(null, 95), "real: median": np.median(real_500), "scrambled: median": np.median(null),
               f"real: best of all {N_TRIALS:,}": real.max()}
display(pd.DataFrame(rows).T)

In [ ]:
print("Leave one year out on the scrambled data: what a search finds when there is nothing to find")
null_loyo = {}
for m in MODES:
    t = leave_one_year_out(null_trials, m, MIN_TRADES)
    null_loyo[m] = t
real_rows = {m: {"real: winners' training": loyo[m]["train"].mean(), "real: winners' held out": loyo[m]["test"].mean(),
                 "scrambled: winners' training": null_loyo[m]["train"].mean(), "scrambled: winners' held out": null_loyo[m]["test"].mean()} for m in MODES}
display(pd.DataFrame(real_rows).T)

## 12. How to read the results

- **Section 5 and 6** are in-sample and inflated by the search. Compare the best with the 99th percentile and with the luck baseline of section 11.
- **Section 7** is the real test inside the development period: the gap between the winners' training and held-out scores is what the search overfitted. If the held-out score is no better than the median trial or the paper's rule, the search found nothing that lasts.
- **Section 8** asks for a result that survives a change of market, at the cost of independence in time.
- **Section 9 and 10** say whether the winner sits on a plateau and which parameters the search can actually distinguish.
- **Costs:** WDO's costs relative to what a contract controls are about twice WIN's, so the joint score (the worse of the two) is usually set by WDO.
- The final test period has not been read. Nothing here nominates a configuration for it.